# 错误处理

## 常见错误和观察

### 错误和底层工具设计
- 五类常见错误：
    1. `SyntaxError`，解析错误，代码不符合Python语法
    2. `NameError`，执行到某个变量时，当前可查找的命名空间中都找不到这个变量
    3. `TypeError`，当前操作不接受这种类型
    4. `IndexError`，索引访问序列时超出有效范围
    5. `AttributeError`，对象没有被访问的那个属性或方法
- 其中 `SyntaxError` 阻断解析；另外四类错误是在程序运行到具体操作时，由名称、类型、索引或属性规则触发

| 工具 | 主要工作 |
|---|---|
| Parser 解析器 | 分析源代码的语法结构，生成结构化表示 |
| Compiler 编译器 | 将源代码或中间表示转换为机器码、字节码等,交给CPU执行 |
| Interpreter 解释器 | 在运行时执行源代码或中间代码 |
| Debugger 调试器 | 控制程序执行，并查看变量、调用栈和运行状态 |

- 这些属于程序运行机制的拓展知识，了解基本定位即可
- 编译与解释不是绝对二分，而是连续谱：

`更多提前编译 ← C —— Java —— Python → 更多运行时解释`

- 连续谱上越偏 Compiler，通常运行效率越高；越偏 Interpreter，通常运行时动态性越强

### 状态观测&暴露错误
- 通过 `print` 和 `f-string` 观察状态
- 利用调试器，让程序运行到某处时，把当时的值暴露出来

In [1]:
price = '12.5'
print(f'price = {price!r},type ={type(price).__name__}')

price = '12.5',type =str


- `!r` 即 `repr()`，是一种转换标记
    - 另外两种常见转换标记是：
        1. `!s`：调用 `str()`
        2. `!a`：调用 `ascii()`
- `!r` 主要出现在 f-string 中，`f"{expression!r}"`
- `repr()`返回一个字符串，会尽量保留能体现对象真实结构的信息
    - 字符串：显示引号和转义字符
    - 数字、布尔值、`None`：代码中的字面量形式
    - 容器：显示容器结构，并递归使用元素的 `repr()`
    - 自定义对象：由类的 `__repr__()` 决定
- `type()` 先返回类对象，类对象的`__name__`就是类名
- 这里避免直接返回一个类对象，而是返回一个类名，方便debug

## 使用IDE和pdb进行基础调试
- 进入方式和交互界面不同：
    - IDE 调试主要通过 GUI、断点和 Debug Console 操作
    - `pdb` 主要通过代码调用或终端命令操作
- 都是暂停-检查-执行部分代码，最后都是为了找出问题
    - 暂停真实运行的 Python 进程
    - 检查
        当前变量、当前栈帧（current stack frame）、调用栈（call stack）、执行位置、表达式结果等
    - 决定执行多少代码

### IDE基础调试（VS Code）
#### 设置断点和启动调试
- 进入脚本，在需要的代码行号左侧打断点
    - 断点行代表尚未执行，但准备执行的语句行
- 按 F5 ，或者打开左侧"运行和调试"面板点击对应 GUI 按钮
- 执行 Debug运行 后，程序从第一行开始运行，执行到断点时暂停
    - 函数定义时只执行def行，定义函数对象并绑定到对应变量
    - 当调用时才执行函数体，调用一次执行一遍

#### 调试的相关区域

| 区域 | 用途 | 查看、操作渠道 |
|---|---|---|
| 变量 | 查看当前选中栈帧中的变量和值 | 直接在变量面板中展开、查看 |
| 监视 | 持续计算并观察指定表达式 | 在 Watch 中添加表达式，如 `price + fee`、`type(price)` |
| 调用堆栈 | 查看当前仍未结束的调用链，并切换不同栈帧 | 点击不同栈帧，查看对应调用现场 |
| 断点 | 管理程序在哪些代码位置暂停 | 查看代码行左侧设置的断点，可以在这里启用、禁用、删除 |
| 调试控制台 | 在当前选中栈帧的上下文中计算表达式、查看变量，也可以临时修改当前运行状态 | 从下面板进入，进入后直接输入 Python 表达式或赋值语句，如 `price`、`price = 100` |




#### 基础操作按钮
- Continue，继续：从当前位置开始连续执行，直至遇到下一个断点、未处理异常、程序终止
- Step Over，逐过程/单步越过：执行当前行，停在下一行；如果当前行调用了函数，不进入函数体，直接执行全部函数体代码，然后停在调用行的下一行
- Step Into，单步调试/单步进入：执行当前行，停在下一行；如果当前行调用了函数，进入函数体，并停在函数体的第一行（不执行函数体代码）
- Step Out，单步跳出：继续执行函数体的剩余代码，停在调用行
    - 注意Step Out虽然停在调用行，但不是回到这行开头，而是回到这行里“函数调用结束之后”的位置（收拾返回值，结束行等等）
    - Step In一直点也会回到调用行，同理也是回到这行里“函数调用结束之后”的位置
- 此外还有重启和中止

### pdb基础调试
#### 进入方式
- 在代码里放暂停点，`breakpoint()`或者`pdb.set_trace()`，程序执行到暂停点后，终端会出现 (Pdb)
- 用pdb启动，`.venv\Scripts\python.exe -m pdb main.py`，会在程序刚开始时进入 (Pdb)

#### 常见命令
| 命令 | 作用 | 后果 |
|---|---|---|
| `p 表达式` | 显示表达式结果 | 会计算表达式；调用函数时可能有副作用 |
| `pp 表达式` | 更整齐地显示复杂对象 | 与 `p` 类似 |
| `l` | 显示当前位置附近源码 | 不执行代码 |
| `ll` | 显示当前函数或栈帧的全部源码 | 不执行代码 |
| `w` | 显示调用栈 | 不执行代码 |
| `n` | 执行到当前函数的下一行 | 会真实执行代码，可能修改状态或抛错 |
| `s` | 执行当前行并尽可能进入被调用函数 | 会真实执行代码 |
| `r` | 继续到当前函数返回 | 可能一次执行很多行 |
| `c` | 继续到下一个断点 | 中间代码全部真实执行 |
| `b` | 查看或设置断点 | 不立即执行代码 |
| `cl` | 清除断点 | 后续运行不会再停在被清除位置 |
| `q` | 退出调试器 | 直接中止被调试程序 |